Earning-Events Analysis Begins Now

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px



In [4]:
# CONSTITUENT-SPECIFIC EVENT MOVEMENT vs NORMAL MOVEMENT
stocks = pd.read_csv(
    "../data/processed/constituent_stocks_data.csv"
)

event_window = pd.read_csv(
    "../data/processed/earnings_event_window.csv"
)

stocks["date"] = pd.to_datetime(stocks["date"])
event_window["market_date"] = pd.to_datetime(
    event_window["market_date"]
)

stocks = stocks.sort_values(
    ["symbol", "date"]
).copy()


# Calculate daily stock return
stocks["daily_return_pct"] = (
    stocks.groupby("symbol")["close"]
    .pct_change()
    * 100
)

stocks["abs_daily_return_pct"] = (
    stocks["daily_return_pct"].abs()
)


# Identifyingonly T0 to T+3 event observations
event_dates = (
    event_window.loc[
        event_window["relative_day"].between(0, 3),
        ["symbol", "market_date"]
    ]
    .drop_duplicates()
    .rename(columns={"market_date": "date"})
)


# Remove only T0 to T+3 from normal baseline T-3, T-2, T-1 remain normal
normal = (
    stocks.merge(
        event_dates.assign(is_event=True),
        on=["symbol", "date"],
        how="left"
    )
)

normal = normal.loc[
    normal["is_event"].isna()
    & normal["daily_return_pct"].notna()
].copy()


# Constituent-specific normal baseline

normal_summary = (
    normal.groupby("symbol")
    .agg(
        normal_days=("daily_return_pct", "count"),
        normal_mean_abs_return=(
            "abs_daily_return_pct",
            "mean"
        ),
        normal_median_abs_return=(
            "abs_daily_return_pct",
            "median"
        ),
        normal_volatility=(
            "daily_return_pct",
            "std"
        ),
    )
    .reset_index()
)


#  event movement: T0 to T+3
event = event_window.loc[
    event_window["relative_day"].between(0, 3)
].copy()

event["abs_daily_return_pct"] = (
    event["daily_return_pct"].abs()
)


event_summary = (
    event.groupby(
        ["symbol", "relative_day"]
    )
    .agg(
        event_days=("daily_return_pct", "count"),
        event_mean_abs_return=(
            "abs_daily_return_pct",
            "mean"
        ),
        event_median_abs_return=(
            "abs_daily_return_pct",
            "median"
        ),
        event_volatility=(
            "daily_return_pct",
            "std"
        ),
    )
    .reset_index()
)


#Comparing each stock with its own normal baseline

event_analysis = event_summary.merge(
    normal_summary,
    on="symbol",
    how="left"
)

event_analysis["movement_increase_pct"] = (
    (
        event_analysis["event_median_abs_return"]
        / event_analysis["normal_median_abs_return"]
    )
    - 1
) * 100



event_analysis = event_analysis[
    [
        "symbol",
        "relative_day",

        "normal_median_abs_return",
        "normal_mean_abs_return",
        "normal_volatility",

        "event_median_abs_return",
        "event_mean_abs_return",
        "event_volatility",

        "movement_increase_pct",
    ]
].sort_values(
    ["symbol", "relative_day"]
)


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\n" + "-" * 80)
print("Constituent specific earnings event impact")
print("-" * 80)

print(
    "\nDoes each constituent move more than its own normal level "
    "during T0 to T+3?"
)
print("\n")
print(
    event_analysis.round(3).to_string(index=False)
)


ANALYSIS 1 — CONSTITUENT-SPECIFIC EARNINGS EVENT IMPACT

Question:
Does each constituent move more than its own normal level during T0 → T+3?

Normal = all historical trading days except T0 → T+3.
T-3, T-2 and T-1 are therefore treated as NORMAL.


    symbol  relative_day  normal_median_abs_return  normal_mean_abs_return  normal_volatility  event_median_abs_return  event_mean_abs_return  event_volatility  movement_increase_pct
    AUBANK             0                     0.955                   1.286              1.800                    1.930                  2.106             2.696                102.004
    AUBANK             1                     0.955                   1.286              1.800                    1.937                  3.359             4.993                102.752
    AUBANK             2                     0.955                   1.286              1.800                    0.568                  1.408             2.295                -40.548
    AUBANK        

**(Lets Findout the relationship of BankNifty with the constituent events and their weights)**

In [9]:
banknifty = pd.read_csv(
    "../data/processed/bank_nifty_daily_amended.csv"
)

stocks = pd.read_csv(
    "../data/processed/constituent_stocks_data.csv"
)

event_window = pd.read_csv(
    "../data/processed/earnings_event_window.csv"
)

weights = pd.read_csv(
    "../data/processed/daily_weight_reconstruction_new.csv"
)


banknifty["date"] = pd.to_datetime(banknifty["date"])
stocks["date"] = pd.to_datetime(stocks["date"])
event_window["date"] = pd.to_datetime(event_window["date"])
event_window["market_date"] = pd.to_datetime(
    event_window["market_date"]
)
event_window["earnings_date"] = pd.to_datetime(
    event_window["earnings_date"]
)
weights["date"] = pd.to_datetime(weights["date"])


banknifty = banknifty.sort_values("date").copy()

banknifty["banknifty_return_pct"] = (
    banknifty["close"].pct_change() * 100
)

banknifty["banknifty_abs_return_pct"] = (
    banknifty["banknifty_return_pct"].abs()
)


stocks = stocks.sort_values(
    ["symbol", "date"]
).copy()

stocks["daily_return_pct"] = (
    stocks.groupby("symbol")["close"].pct_change() * 100
)

weights["regime_id"] = np.select(
    [
        weights["date"] < pd.Timestamp("2024-09-30"),
        weights["date"] < pd.Timestamp("2025-12-31"),
    ],
    [
        "R1",
        "R2",
    ],
    default="R3",
)

regime_structure = (
    weights.groupby(
        ["regime_id", "date"]
    )["symbol"]
    .nunique()
    .groupby(level=0)
    .agg(
        min_constituents="min",
        max_constituents="max",
        trading_days="count",
    )
    .reset_index()
)

print("-" * 80)
print("GROUP 1 — CONFIGURATION VALIDATION")
print("-" * 80)

print("\nRegime structure:")
print(
    regime_structure.to_string(index=False)
)

print("\nWeight coverage:")
print(
    weights.groupby("regime_id")["weight"]
    .agg(
        mean="mean",
        min="min",
        max="max",
    )
    .round(4)
    .to_string()
)


# Narrowing down to  T0 AND T+1
events = event_window.loc[
    event_window["relative_day"].isin([0, 1])
].copy()


# Unique event date/relative day mapping
event_dates = (
    events[
        [
            "market_date",
            "relative_day",
        ]
    ]
    .drop_duplicates()
)


print("\nEvent observations:", len(events))
print(
    "Unique earnings events:",
    events["event_id"].nunique()
)
print(
    "T0 events:",
    (events["relative_day"] == 0).sum()
)
print(
    "T+1 events:",
    (events["relative_day"] == 1).sum()
)

GROUP 1 — CONFIGURATION VALIDATION

Regime structure:
regime_id  min_constituents  max_constituents  trading_days
       R1                12                12           428
       R2                12                12           310
       R3                14                14           164

Weight coverage:
             mean     min      max
regime_id                         
R1         8.3333  0.7974  30.6513
R2         8.3333  1.6185  30.3053
R3         7.1429  0.8000  25.1400

Event observations: 450
Unique earnings events: 225
T0 events: 225
T+1 events: 225


In [10]:
# Does Bank Nifty itself show an event-period movement shock?

event_market_dates = (
    events["market_date"]
    .drop_duplicates()
)

banknifty_q1 = banknifty.copy()

banknifty_q1["event_day"] = (
    banknifty_q1["date"]
    .isin(event_market_dates)
)


# Normal Banknifty baseline
normal = banknifty_q1.loc[
    ~banknifty_q1["event_day"]
    & banknifty_q1["banknifty_return_pct"].notna()
]


normal_stats = {
    "observations": len(normal),
    "mean_return_pct": normal[
        "banknifty_return_pct"
    ].mean(),
    "median_abs_return_pct": normal[
        "banknifty_abs_return_pct"
    ].median(),
    "mean_abs_return_pct": normal[
        "banknifty_abs_return_pct"
    ].mean(),
    "volatility_pct": normal[
        "banknifty_return_pct"
    ].std(),
}

# T0 & T-1
event_banknifty = (
    banknifty_q1[
        banknifty_q1["event_day"]
    ]
    .merge(
        event_dates,
        left_on="date",
        right_on="market_date",
        how="inner",
    )
)


event_summary = (
    event_banknifty
    .groupby("relative_day")
    .agg(
        observations=(
            "banknifty_return_pct",
            "count",
        ),
        mean_return_pct=(
            "banknifty_return_pct",
            "mean",
        ),
        median_return_pct=(
            "banknifty_return_pct",
            "median",
        ),
        mean_abs_return_pct=(
            "banknifty_abs_return_pct",
            "mean",
        ),
        median_abs_return_pct=(
            "banknifty_abs_return_pct",
            "median",
        ),
        volatility_pct=(
            "banknifty_return_pct",
            "std",
        ),
    )
    .reset_index()
)


event_summary["median_abs_increase_vs_normal_pct"] = (
    (
        event_summary["median_abs_return_pct"]
        / normal_stats["median_abs_return_pct"]
    )
    - 1
) * 100


print("-" * 70)
print("Banknifty Normal v/s Event effect")
print("-" * 70)

print("\nNormal baseline:")
print(
    pd.Series(normal_stats)
    .round(4)
    .to_string()
)

print("\nT0/T+1:")
print(
    event_summary.round(4)
    .to_string(index=False)
)

--------------------------------------------------------------------------------
Banknifty Normal v/s Event effect
--------------------------------------------------------------------------------

Normal baseline:
observations             695.0000
mean_return_pct            0.0576
median_abs_return_pct      0.4560
mean_abs_return_pct        0.6782
volatility_pct             0.9951

T0/T+1:
 relative_day  observations  mean_return_pct  median_return_pct  mean_abs_return_pct  median_abs_return_pct  volatility_pct  median_abs_increase_vs_normal_pct
            0           135          -0.0057             0.0276               0.7229                 0.5794          0.9406                            27.0681
            1           135          -0.1344            -0.1892               0.6979                 0.5809          0.9167                            27.3880


In [11]:
#How much of actual Bank Nifty movement is explained by constituents weights

# Attaching daily weights to all constituent returns

weighted_stocks = stocks.merge(
    weights[
        [
            "date",
            "symbol",
            "weight",
            "regime_id",
        ]
    ],
    on=["date", "symbol"],
    how="inner",
)


# Approximate constituent contribution
weighted_stocks["weighted_return_pct"] = (
    weighted_stocks["weight"]
    / 100
    * weighted_stocks["daily_return_pct"]
)


# Reconstruct Bank Nifty daily return
reconstructed = (
    weighted_stocks
    .groupby(
        ["date", "regime_id"]
    )
    .agg(
        reconstructed_return_pct=(
            "weighted_return_pct",
            "sum",
        ),
        constituent_count=(
            "symbol",
            "nunique",
        ),
    )
    .reset_index()
)


# Attach actual Bank Nifty
reconstructed = reconstructed.merge(
    banknifty[
        [
            "date",
            "banknifty_return_pct",
        ]
    ],
    on="date",
    how="inner",
)


# Reconstruction residual
reconstructed["residual_pct"] = (
    reconstructed["banknifty_return_pct"]
    - reconstructed["reconstructed_return_pct"]
)

reconstructed["abs_residual_pct"] = (
    reconstructed["residual_pct"].abs()
)


# Event status
reconstructed["relative_day"] = np.nan

for relative_day in [0, 1]:

    dates = set(
        events.loc[
            events["relative_day"] == relative_day,
            "market_date",
        ]
    )

    reconstructed.loc[
        reconstructed["date"].isin(dates),
        "relative_day",
    ] = relative_day


# Overall reconstruction quality
q2_overall = pd.Series(
    {
        "observations": len(reconstructed),
        "mean_actual_return_pct":
            reconstructed[
                "banknifty_return_pct"
            ].mean(),
        "mean_reconstructed_return_pct":
            reconstructed[
                "reconstructed_return_pct"
            ].mean(),
        "mean_residual_pct":
            reconstructed[
                "residual_pct"
            ].mean(),
        "median_residual_pct":
            reconstructed[
                "residual_pct"
            ].median(),
        "mean_abs_residual_pct":
            reconstructed[
                "abs_residual_pct"
            ].mean(),
        "median_abs_residual_pct":
            reconstructed[
                "abs_residual_pct"
            ].median(),
        "residual_std_pct":
            reconstructed[
                "residual_pct"
            ].std(),
        "correlation":
            reconstructed[
                [
                    "banknifty_return_pct",
                    "reconstructed_return_pct",
                ]
            ].corr().iloc[0, 1],
    }
)


print("-" * 70)
print("Weighted constituent v/s Actual Banknifty Movement")
print("-" * 70)

print(
    q2_overall.round(5).to_string()
)

print("\nBy regime:")
print(
    reconstructed
    .groupby("regime_id")
    .agg(
        observations=(
            "date",
            "count",
        ),
        mean_residual=(
            "residual_pct",
            "mean",
        ),
        median_residual=(
            "residual_pct",
            "median",
        ),
        mean_abs_residual=(
            "abs_residual_pct",
            "mean",
        ),
        median_abs_residual=(
            "abs_residual_pct",
            "median",
        ),
    )
    .round(5)
    .to_string()
)

----------------------------------------------------------------------
Weighted constituent v/s Actual Banknifty Movement
----------------------------------------------------------------------
observations                     902.00000
mean_actual_return_pct             0.03603
mean_reconstructed_return_pct      0.05118
mean_residual_pct                 -0.01520
median_residual_pct               -0.00685
mean_abs_residual_pct              0.03434
median_abs_residual_pct            0.01435
residual_std_pct                   0.11947
correlation                        0.99251

By regime:
           observations  mean_residual  median_residual  mean_abs_residual  median_abs_residual
regime_id                                                                                      
R1                  428       -0.01597         -0.00661            0.03071              0.01299
R2                  310       -0.02005         -0.00565            0.03075              0.01243
R3                  164 

In [13]:
# Residual: Normal vs T0 vs T+1

# Keep T0 and T+1 as independent event-date flags
t0_dates = set(
    events.loc[events["relative_day"] == 0, "market_date"].unique()
)

t1_dates = set(
    events.loc[events["relative_day"] == 1, "market_date"].unique()
)

reconstructed["is_t0_event"] = reconstructed["date"].isin(t0_dates)
reconstructed["is_t1_event"] = reconstructed["date"].isin(t1_dates)
reconstructed["is_event"] = (
    reconstructed["is_t0_event"] | reconstructed["is_t1_event"]
)

# Valid residual observations only
valid = reconstructed["residual_pct"].notna()

normal = reconstructed[
    (~reconstructed["is_event"]) & valid
]["residual_pct"]

t0 = reconstructed[
    reconstructed["is_t0_event"] & valid
]["residual_pct"]

t1 = reconstructed[
    reconstructed["is_t1_event"] & valid
]["residual_pct"]


def residual_summary(x):
    return {
        "observations": len(x),
        "mean_residual_pct": x.mean(),
        "median_residual_pct": x.median(),
        "mean_abs_residual_pct": x.abs().mean(),
        "median_abs_residual_pct": x.abs().median(),
        "residual_std_pct": x.std()
    }


q3 = pd.DataFrame({
    "Normal": residual_summary(normal),
    "T0": residual_summary(t0),
    "T+1": residual_summary(t1)
}).T

normal_median_abs = q3.loc["Normal", "median_abs_residual_pct"]

q3["median_abs_change_vs_normal_pct"] = (
    (q3["median_abs_residual_pct"] / normal_median_abs - 1) * 100
)

print("T0 unique dates:", len(t0_dates))
print("T+1 unique dates:", len(t1_dates))
print("T0 + T+1 overlapping dates:", len(t0_dates & t1_dates))
print("\nQ3 — Residual behaviour")
display(q3.round(4))

T0 unique dates: 135
T+1 unique dates: 135
T0 ∩ T+1 overlapping dates: 58

Q3 — Residual behaviour


,observations,mean_residual_pct,median_residual_pct,mean_abs_residual_pct,median_abs_residual_pct,residual_std_pct,median_abs_change_vs_normal_pct
Normal,689.0,-0.0150,-0.0061,0.0299,0.0133,0.0974,0.0000
T0,135.0,-0.0187,-0.0105,0.0624,0.0184,0.2153,38.7711
T+1,135.0,-0.0068,-0.0116,0.0525,0.0184,0.2029,38.7711


In [19]:
# What information is useful for predicting Bank Nifty movement?

#event-level dataset
event_features = events[
    [
        "symbol",
        "market_date",
        "relative_day",
        "earnings_eps_surprise",
        "earnings_revenue_surprise",
        "daily_return_pct"
    ]
].drop_duplicates().copy()

event_features["earnings_eps_surprise"] = pd.to_numeric(
    event_features["earnings_eps_surprise"],
    errors="coerce"
)

event_features["earnings_revenue_surprise"] = pd.to_numeric(
    event_features["earnings_revenue_surprise"],
    errors="coerce"
)

event_features = event_features.merge(
    weighted_stocks[
        [
            "date",
            "symbol",
            "weight",
            "weighted_return_pct"
        ]
    ],
    left_on=["market_date", "symbol"],
    right_on=["date", "symbol"],
    how="left"
).drop(columns="date")

# Adding Banknifty return
event_features = event_features.merge(
    reconstructed[
        [
            "date",
            "banknifty_return_pct"
        ]
    ],
    left_on="market_date",
    right_on="date",
    how="left"
).drop(columns="date")

# Absolute movement variables
event_features["stock_abs_return_pct"] = event_features["daily_return_pct"].abs()

event_features["banknifty_abs_return_pct"] = (
    event_features["banknifty_return_pct"].abs()
)

# Variables to test
predictors = [
    "earnings_eps_surprise",
    "earnings_revenue_surprise",
    "daily_return_pct",
    "stock_abs_return_pct",
    "weight",
    "weighted_return_pct"
]

targets = {
    "BankNifty_Return": "banknifty_return_pct",
    "BankNifty_Abs_Return": "banknifty_abs_return_pct"
}

# Spearman correlation Spearman = Pearson correlation of ranks
event_features_results = []

for relative_day in [0, 1]:

    subset = event_features[
        event_features["relative_day"] == relative_day
    ]

    for predictor in predictors:

        for target_name, target_col in targets.items():

            pair = subset[
                [predictor, target_col]
            ].dropna()

            if len(pair) < 10:
                continue

            x_rank = pair[predictor].rank(method="average")
            y_rank = pair[target_col].rank(method="average")

            corr = x_rank.corr(y_rank)

            event_features_results.append({
                "relative_day": (
                    "T0" if relative_day == 0 else "T+1"
                ),
                "predictor": predictor,
                "target": target_name,
                "observations": len(pair),
                "spearman_corr": corr
            })

event_features_results = pd.DataFrame(event_features_results)

# Full results
event_features_results["abs_corr"] = (
    event_features_results["spearman_corr"].abs()
)

print("event_features — Full correlation results")

display(
    event_features_results
    .sort_values(
        [
            "relative_day",
            "target",
            "abs_corr"
        ],
        ascending=[True, True, False]
    )
    .drop(columns="abs_corr")
    .round(4)
)

# Strongest relationship for each target/day
event_features_summary = (
    event_features_results
    .sort_values("abs_corr", ascending=False)
    .groupby(
        ["relative_day", "target"],
        as_index=False
    )
    .first()
    .drop(columns="abs_corr")
)

print("\nevent_features — Strongest relationship by target/day")

display(
    event_features_summary.round(4)
)

Q4 — Full correlation results


,relative_day,predictor,target,observations,spearman_corr
19,T+1,stock_abs_return_pct,BankNifty_Abs_Return,225,0.3279
23,T+1,weighted_return_pct,BankNifty_Abs_Return,186,0.0825
17,T+1,daily_return_pct,BankNifty_Abs_Return,225,0.0772
13,T+1,earnings_eps_surprise,BankNifty_Abs_Return,224,0.0730
21,T+1,weight,BankNifty_Abs_Return,186,-0.0294
15,T+1,earnings_revenue_surprise,BankNifty_Abs_Return,225,-0.0087
22,T+1,weighted_return_pct,BankNifty_Return,186,0.4166
16,T+1,daily_return_pct,BankNifty_Return,225,0.3932
20,T+1,weight,BankNifty_Return,186,-0.1659
18,T+1,stock_abs_return_pct,BankNifty_Return,225,0.0613



Q4 — Strongest relationship by target/day


,relative_day,target,predictor,observations,spearman_corr
0,T+1,BankNifty_Abs_Return,stock_abs_return_pct,225,0.3279
1,T+1,BankNifty_Return,weighted_return_pct,186,0.4166
2,T0,BankNifty_Abs_Return,stock_abs_return_pct,225,0.2081
3,T0,BankNifty_Return,weighted_return_pct,186,0.4028


In [20]:
# Does an earnings event from a majority-weight constituent group produce a different Banknifty response?


# Creating majority-weight membership for every date
weight_groups = weighted_stocks[
    [
        "date",
        "symbol",
        "weight"
    ]
].copy()

weight_groups = weight_groups.sort_values(
    ["date", "weight"],
    ascending=[True, False]
)

weight_groups["cumulative_weight"] = (
    weight_groups
    .groupby("date")["weight"]
    .cumsum()
)

# A constituent belongs to the majority group if the cumulative weight crosses 50%.
weight_groups["majority_weight_group"] = (
    weight_groups["cumulative_weight"] <= 50
)

# Always include the constituent that crosses 50%
crossing = (
    weight_groups
    .groupby("date")["cumulative_weight"]
    .transform("min")
)

# Identify first constituent whose cumulative weight >= 50%
weight_groups["majority_weight_group"] = (
    weight_groups["cumulative_weight"] <= 50
)

crossing_symbol = (
    weight_groups[
        weight_groups["cumulative_weight"] >= 50
    ]
    .sort_values(["date", "cumulative_weight"])
    .groupby("date")
    .head(1)[["date", "symbol"]]
)

weight_groups = weight_groups.merge(
    crossing_symbol.assign(
        crossing_constituent=True
    ),
    on=["date", "symbol"],
    how="left"
)

weight_groups["crossing_constituent"] = (
    weight_groups["crossing_constituent"]
    .fillna(False)
)

weight_groups["majority_weight_group"] = (
    weight_groups["majority_weight_group"]
    | weight_groups["crossing_constituent"]
)

# Building event-level dataset
df = events[
    [
        "symbol",
        "market_date",
        "relative_day",
        "daily_return_pct"
    ]
].drop_duplicates().copy()

df["stock_abs_return_pct"] = (
    df["daily_return_pct"].abs()
)

# Adding weight-group classification
df = df.merge(
    weight_groups[
        [
            "date",
            "symbol",
            "weight",
            "majority_weight_group"
        ]
    ],
    left_on=["market_date", "symbol"],
    right_on=["date", "symbol"],
    how="left"
).drop(columns="date")

# Adding Banknifty return
df = df.merge(
    reconstructed[
        [
            "date",
            "banknifty_return_pct"
        ]
    ],
    left_on="market_date",
    right_on="date",
    how="left"
).drop(columns="date")

df["banknifty_abs_return_pct"] = (
    df["banknifty_return_pct"].abs()
)

#Comparimg groups
df_summary = (
    df[df["relative_day"].isin([0, 1])]
    .groupby(
        [
            "relative_day",
            "majority_weight_group"
        ]
    )
    .agg(
        observations=("symbol", "size"),
        median_weight=("weight", "median"),
        median_stock_abs_return_pct=(
            "stock_abs_return_pct",
            "median"
        ),
        median_banknifty_abs_return_pct=(
            "banknifty_abs_return_pct",
            "median"
        ),
        mean_banknifty_return_pct=(
            "banknifty_return_pct",
            "mean"
        )
    )
    .reset_index()
)

df_summary["group"] = (
    df_summary["majority_weight_group"]
    .map({
        True: "Majority-weight group",
        False: "Lower-weight group"
    })
)

df_summary["relative_day"] = (
    df_summary["relative_day"]
    .map({
        0: "T0",
        1: "T+1"
    })
)

print("Majority-weight vs lower-weight events")

display(
    df_summary[
        [
            "relative_day",
            "group",
            "observations",
            "median_weight",
            "median_stock_abs_return_pct",
            "median_banknifty_abs_return_pct",
            "mean_banknifty_return_pct"
        ]
    ].round(4)
)


Q5 — Majority-weight vs lower-weight events


,relative_day,group,observations,median_weight,median_stock_abs_return_pct,median_banknifty_abs_return_pct,mean_banknifty_return_pct
0,T0,Lower-weight group,149,3.3047,1.6876,0.6073,-0.0317
1,T0,Majority-weight group,37,23.8695,1.5244,0.5542,0.1408
2,T+1,Lower-weight group,149,3.2986,1.6393,0.5809,-0.0329
3,T+1,Majority-weight group,37,24.1448,0.8404,0.3455,-0.1376


In [21]:
# When a low-weight constituent experiences an unusually large move, how large is its weighted contribution and what happens to Banknifty?


# Normal stock days exclude T0 and T+1 event observations. we use each constituent's own normal distribution.
event_dates = set(
    events.loc[
        events["relative_day"].isin([0, 1]),
        "market_date"
    ].unique()
)

normal_stock = weighted_stocks[
    ~weighted_stocks["date"].isin(event_dates)
].copy()

normal_stock["abs_return_pct"] = (
    normal_stock["daily_return_pct"].abs()
)

# 95th percentile movement for each constituent
stock_extreme_threshold = (
    normal_stock
    .groupby("symbol")["abs_return_pct"]
    .quantile(0.95)
    .rename("normal_p95_abs_return_pct")
    .reset_index()
)


df = events[
    [
        "symbol",
        "market_date",
        "relative_day",
        "daily_return_pct"
    ]
].drop_duplicates().copy()

df["stock_abs_return_pct"] = (
    df["daily_return_pct"].abs()
)

# Adding weight
df = df.merge(
    weighted_stocks[
        [
            "date",
            "symbol",
            "weight",
            "weighted_return_pct"
        ]
    ],
    left_on=["market_date", "symbol"],
    right_on=["date", "symbol"],
    how="left"
).drop(columns="date")

# Adding own normal movement threshold
df = df.merge(
    stock_extreme_threshold,
    on="symbol",
    how="left"
)

# Adding Banknifty response
df = df.merge(
    reconstructed[
        [
            "date",
            "banknifty_return_pct"
        ]
    ],
    left_on="market_date",
    right_on="date",
    how="left"
).drop(columns="date")

df["banknifty_abs_return_pct"] = (
    df["banknifty_return_pct"].abs()
)

# Defining low-weight events
low_weight_cutoff = df["weight"].quantile(0.25)

df["low_weight"] = (
    df["weight"] <= low_weight_cutoff
)

# Extreme = above that bank's own normal 95th percentile
df["extreme_stock_move"] = (
    df["stock_abs_return_pct"]
    > df["normal_p95_abs_return_pct"]
)

# Combined condition
df["extreme_low_weight_event"] = (
    df["low_weight"]
    & df["extreme_stock_move"]
)

# Comparing extreme low-weight events with other events
df_subset = df[
    df["relative_day"].isin([0, 1])
].copy()

df_summary = (
    df_subset
    .groupby("extreme_low_weight_event")
    .agg(
        observations=("symbol", "size"),
        median_weight=("weight", "median"),
        median_stock_abs_return_pct=(
            "stock_abs_return_pct",
            "median"
        ),
        median_weighted_impact_pct=(
            "weighted_return_pct",
            lambda x: x.abs().median()
        ),
        median_banknifty_abs_return_pct=(
            "banknifty_abs_return_pct",
            "median"
        ),
        mean_banknifty_return_pct=(
            "banknifty_return_pct",
            "mean"
        )
    )
    .reset_index()
)

df_summary["event_group"] = (
    df_summary["extreme_low_weight_event"]
    .map({
        True: "Extreme + low-weight",
        False: "Other events"
    })
)

print("Low-weight cutoff:", round(low_weight_cutoff, 4))

print("Extreme low-weight events")

display(
    df_summary[
        [
            "event_group",
            "observations",
            "median_weight",
            "median_stock_abs_return_pct",
            "median_weighted_impact_pct",
            "median_banknifty_abs_return_pct",
            "mean_banknifty_return_pct"
        ]
    ].round(4)
)

# The actual extreme low-weight events
extreme_events = (
    df_subset[
        df_subset["extreme_low_weight_event"]
    ]
    [
        [
            "symbol",
            "market_date",
            "relative_day",
            "weight",
            "stock_abs_return_pct",
            "normal_p95_abs_return_pct",
            "weighted_return_pct",
            "banknifty_return_pct",
            "banknifty_abs_return_pct"
        ]
    ]
    .sort_values(
        "stock_abs_return_pct",
        ascending=False
    )
)

print("\nActual extreme low-weight events:")

display(
    extreme_events.round(4)
)

Low-weight cutoff: 2.6287
Extreme low-weight events


,event_group,observations,median_weight,median_stock_abs_return_pct,median_weighted_impact_pct,median_banknifty_abs_return_pct,mean_banknifty_return_pct
0,Other events,427,4.9838,1.5432,0.0735,0.5950,-0.0329
1,Extreme + low-weight,23,2.0655,4.7244,0.0888,0.6268,0.2393



Actual extreme low-weight events:


C:\Users\musaib\AppData\Local\Temp\ipykernel_17332\858864522.py:185: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  extreme_events.round(4)


,symbol,market_date,relative_day,weight,stock_abs_return_pct,normal_p95_abs_return_pct,weighted_return_pct,banknifty_return_pct,banknifty_abs_return_pct
139,AUBANK,2024-01-29,1,1.9675,11.5137,3.3879,-0.2265,1.2843,1.2843
360,UNIONBANK,2026-01-14,0,1.6992,7.8705,4.5285,0.1337,0.0023,0.0023
26,BANKBARODA,2023-02-03,0,2.5032,6.1993,3.4108,0.1552,2.0418,2.0418
204,PNB,2024-07-29,0,2.0730,5.8775,4.0162,0.1218,0.2150,0.2150
25,PNB,2023-01-31,1,1.3300,5.6213,4.0162,0.0748,0.6626,0.6626
61,FEDERALBNK,2023-07-13,1,2.4375,5.5866,3.2131,-0.1362,0.0573,0.0573
178,BANDHANBNK,2024-05-16,0,1.0246,5.4239,4.3553,-0.0556,0.6073,0.6073
314,CANBK,2025-07-24,0,2.3088,5.2968,3.8179,0.1223,-0.2524,0.2524
247,AUBANK,2025-01-27,1,2.5417,5.2468,3.3879,-0.1334,-0.6268,0.6268
252,CANBK,2025-01-27,0,2.2507,4.9819,3.8179,-0.1121,-0.6268,0.6268
